In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from torchvision import datasets

# -----------------------------------------------------------------
# [점검 및 복구 1] 장치 선언부 (device)
# -----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("현재 사용 장치:", device)

# 베이지안 최적화 라이브러리 자동 설치 및 임포트
try:
    from bayes_opt import BayesianOptimization
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "bayesian-optimization"])
    from bayes_opt import BayesianOptimization

# -----------------------------------------------------------------
# [점검 및 복구 2] 원본 데이터 로드 및 변수 선언부 (train_images 등)
# -----------------------------------------------------------------
# 메모리 유실을 방지하기 위해 datasets에서 MNIST를 직접 받아 넘파이 배열로 복구합니다.
mnist_train = datasets.MNIST(root='./data', train=True, download=True)
mnist_test = datasets.MNIST(root='./data', train=False, download=True)

train_images = mnist_train.data.numpy()
train_labels = mnist_train.targets.numpy()
test_images = mnist_test.data.numpy()
test_labels = mnist_test.targets.numpy()

# 사용자님 파일의 가공 방식 반영 (float32 변환 및 255 나누기)
X_train_processed = np.array(train_images, dtype=np.float32) / 255.0
Y_train_processed = np.array(train_labels, dtype=np.int64)

X_test_processed = np.array(test_images, dtype=np.float32) / 255.0
Y_test_processed = np.array(test_labels, dtype=np.int64)

# MNIST 2D CNN을 위해 채널 차원(1)을 unsqueeze로 추가하여 텐서 변환
full_train_tensor_X = torch.tensor(X_train_processed, dtype=torch.float32).unsqueeze(1)
full_train_tensor_Y = torch.tensor(Y_train_processed, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).unsqueeze(1).to(device)
Y_test_tensor = torch.tensor(Y_test_processed, dtype=torch.long).to(device)

full_train_dataset = TensorDataset(full_train_tensor_X, full_train_tensor_Y)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

# [진짜 검증 데이터 분리] 훈련 80%(48000개) : 검증 20%(12000개)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


# -----------------------------------------------------------------
# [점검 및 복구 3] 원본 모델 클래스 선언부 (Net)
# -----------------------------------------------------------------
# 사용자님 파일에 있던 Conv2d -> ReLU -> MaxPool2d 기반의 실제 구조입니다.
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 28x28 크기가 MaxPool을 두 번 거치면 7x7이 됩니다.
        self.fc1 = nn.Linear(in_features=64 * 7 * 7, out_features=128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(in_features=128, out_features=10) # 0~9 손글씨 분류용

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = x.view(x.size(0), -1) # 차원 평탄화(Flatten)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        return x


# -----------------------------------------------------------------
# [점검 및 복구 4] 원본 정확도 계산 함수 선언부 (calculate_accuracy)
# -----------------------------------------------------------------
def calculate_accuracy(logits, labels):
    predictions = torch.argmax(logits, dim=1)
    correct = (predictions == labels).float()
    return correct.mean()


# -----------------------------------------------------------------
# 5. 베이지안 최적화 전용 목적 함수 정의
# -----------------------------------------------------------------
def train_and_evaluate(lr, batch_size_log):
    current_lr = lr
    current_batch_size = int(2 ** int(batch_size_log)) # 로그스케일을 실제 배치 사이즈로 변환

    train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=current_batch_size, shuffle=False)

    # [복구 확인] 잘못 들어가 있던 FruitClassifier 대신 원본 Net()을 정확히 선언합니다.
    local_model = Net().to(device)
    local_criterion = nn.CrossEntropyLoss()
    local_optimizer = optim.Adam(local_model.parameters(), lr=current_lr)

    best_val_loss = float('inf')
    best_val_acc = 0.0

    # 하이퍼파라미터 조합 하나당 5에포크씩 테스트하여 최적의 조합을 찾습니다.
    search_epochs = 5

    for epoch in range(search_epochs):
        # 학습 모드
        local_model.train()
        for batch_X, batch_Y in train_loader:
            batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)

            logits = local_model(batch_X)
            loss = local_criterion(logits, batch_Y)

            local_optimizer.zero_grad()
            loss.backward()
            local_optimizer.step()

        # 검증 모드 (오염되지 않은 격리된 검증 데이터셋 활용)
        local_model.eval()
        val_loss = 0.0
        val_correct = 0
        total_val_samples = 0

        with torch.no_grad():
            for batch_X, batch_Y in val_loader:
                batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
                logits = local_model(batch_X)

                loss = local_criterion(logits, batch_Y)
                val_loss += loss.item() * batch_X.size(0)

                acc = calculate_accuracy(logits, batch_Y)
                val_correct += acc.item() * batch_X.size(0)
                total_val_samples += batch_X.size(0)

        epoch_val_loss = val_loss / total_val_samples
        epoch_val_acc = val_correct / total_val_samples

        # [베스트 모델 체크포인트 저장] 최고 성능 시점에 가중치 저장
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_val_acc = epoch_val_acc
            torch.save(local_model.state_dict(), 'mnist_best_temp_model.pth')

    return best_val_acc


# -----------------------------------------------------------------
# 6. 베이지안 최적화 세팅 및 튜닝 가동
# -----------------------------------------------------------------
pbounds = {
    'lr': (0.0005, 0.005),
    'batch_size_log': (5, 7) # 2^5(32) ~ 2^7(128) 범위 탐색
}

optimizer_bo = BayesianOptimization(
    f=train_and_evaluate,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

print("\n=== [보고] 복구 완료 및 베이지안 최적화 탐색을 시작합니다 ===")
optimizer_bo.maximize(init_points=2, n_iter=5)


# -----------------------------------------------------------------
# 7. 최종 하이퍼파라미터 조합 출력 및 실전(Test) 데이터 최종 평가
# -----------------------------------------------------------------
best_target = optimizer_bo.max['target']
best_config = optimizer_bo.max['params']
optimal_lr = best_config['lr']
optimal_batch_size = int(2 ** int(best_config['batch_size_log']))

print("\n=======================================================")
print("=== [보고] 최종 점검 결과: 모든 선언부 복구 및 연산 성공 ===")
print("=======================================================")
print(f"최적의 Learning Rate: {optimal_lr:.5f}")
print(f"최적의 Batch Size: {optimal_batch_size}")
print(f"최고 검증(Validation) 정확도: {best_target * 100:.2f}%")

# 최적화 도중 저장된 가장 똑똑한 베스트 가중치 파일을 최종 실전 테스트에 반영
final_evaluated_model = Net().to(device)
if os.path.exists('mnist_best_temp_model.pth'):
    final_evaluated_model.load_state_dict(torch.load('mnist_best_temp_model.pth'))

final_evaluated_model.eval()
test_correct = 0

with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
        logits = final_evaluated_model(batch_X)

        predictions = torch.argmax(logits, dim=1)
        test_correct += (predictions == batch_Y).sum().item()

final_test_accuracy = test_correct / len(test_dataset)
print(f"★ 최종 최적화가 완료된 모델의 실전(Test) 정확도: {final_test_accuracy * 100:.2f}%")